In [ ]:
import  pandas as pd
import  numpy as np
import  random
import  os
import  json
import  matplotlib.pyplot as plt
import  seaborn as sns
from  datetime  import  datetime, timedelta

RF01 – Criar ou Carregar o Dataset de Vendas

In [ ]:
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta

def gerar_dataset_vendas(n_registros=150, seed=42):
    """  Gera um dataset sintético de vendas com dados  propositalmente sujos,
    incluindo valores nulos, strings sujas, datas inválidas e outliers.  """
    random.seed(seed)
    np.random.seed(seed)
    produtos = ['Notebook', 'Smartphone', 'Tablet', 'Monitor', 'Teclado',
                'Mouse']
    precos = {'Notebook': 3500, 'Smartphone': 2200, 'Tablet': 1800,
                'Monitor': 1200, 'Teclado': 250, 'Mouse': 120}
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                "Tablet": "Celulares", "Monitor": "Computadores", "Teclado": "Periféricos",
                "Mouse": "Periféricos"}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    clientes = [f"Cliente_{i:03d}" for i in range(1, 31)]
    data_inicio = datetime(2024, 1, 1)
    dados = []
    for i in range(n_registros):
        produto = random.choice(produtos)
        quantidade = random.randint(1, 10)
        preco = precos[produto]
        data = data_inicio + timedelta(days=random.randint(0, 364))

        # Inserindo dados intencionalmente sujos para limpeza
        if random.random() < 0.05:
            quantidade = None # valor nulo
        if random.random() < 0.04:
            preco = None # valor nulo
        if random.random() < 0.03:
            produto = "  " + produto  # espaço extra  (string suja)
        data_str = data.strftime("%Y-%m-%d") if random.random() > 0.02 else "DATA INVALIDA"

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_str,
            "cliente": random.choice(clientes),
            "produto": produto,
            "categoria": categorias.get(produto.strip(), "Outros"),
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })
    return pd.DataFrame(dados)

# Gerar e salvar
df_bruto = gerar_dataset_vendas()
os.makedirs('data/raw', exist_ok=True)  # Adicionado  para garantir que o
# diretório exista
df_bruto.to_csv("data/raw/vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
print(df_bruto.head())
# Alternativa: usar CSV externo

Dataset gerado com 150 registros.
   id_venda     data_venda      cliente   produto     categoria        regiao  \
0         1     2024-01-13  Cliente_024     Mouse   Periféricos         Norte   
1         2     2024-08-04  Cliente_018  Notebook  Computadores           Sul   
2         3  DATA INVALIDA  Cliente_026     Mouse   Periféricos           Sul   
3         4     2024-06-23  Cliente_013     Mouse   Periféricos       Sudeste   
4         5     2024-11-05  Cliente_030    Tablet     Celulares  Centro-Oeste   

   quantidade  preco_unitario  
0         2.0           120.0  
1         NaN          3500.0  
2         9.0           120.0  
3         7.0           120.0  
4         6.0          1800.0  


RF02 – Inspecionar e Descrever os Dados

In [ ]:
 def inspecionar_dados(df):
  """Exibe informações básicas do DataFrame."""
  print("\n=== INSPEÇÃO INICIAL DO DATASET ===")
  print(f"Shape: {df.shape}")
  print(f"\nColunas: {list(df.columns)}")
  print(f"\nTipos de dados:\n{df.dtypes}")
  print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
  print(f"\nPrimeiros registros:\n{df.head()}")
  print(f"\nEstatísticas descritivas:\n{df.describe()}")
  return df.describe(include="all")

inspecionar_dados(df_bruto)


=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (150, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        5
preco_unitario    2
dtype: int64

Primeiros registros:
   id_venda     data_venda      cliente   produto     categoria        regiao  \
0         1     2024-01-13  Cliente_024     Mouse   Periféricos         Norte   
1         2     2024-08-04  Cliente_018  Notebook  Computadores           Sul   
2         3  DATA INVALIDA  Cliente_026     Mouse   Periféricos           Sul   
3         4     2024-06-23  Cliente_013     Mous

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
count,150.000000,150,150,150,150,150,145.000000,148.000000
unique,NaN,117,30,8,3,5,NaN,NaN
top,NaN,DATA INVALIDA,Cliente_018,Mouse,Celulares,Sudeste,NaN,NaN
freq,NaN,4,8,28,51,41,NaN,NaN
mean,75.500000,NaN,NaN,NaN,NaN,NaN,5.468966,1558.513514
std,43.445368,NaN,NaN,NaN,NaN,NaN,2.808853,1190.199414
min,1.000000,NaN,NaN,NaN,NaN,NaN,1.000000,120.000000
25%,38.250000,NaN,NaN,NaN,NaN,NaN,3.000000,250.000000
50%,75.500000,NaN,NaN,NaN,NaN,NaN,5.000000,1800.000000
75%,112.750000,NaN,NaN,NaN,NaN,NaN,8.000000,2200.000000


 RF03 – Limpar e Tratar os Dados

In [ ]:
import re
import os
import pandas as pd # Adding explicit import for pandas as it's used extensively

# LIMPEZA COM EXPRESSÕES REGULARES (módulo re)
# re.sub(padrão, substituto, string) substitui todas as ocorrências
# do padrão pela string substituta.
# r"\s+" é um padrão regex que significa "um ou mais espaços em branco"
# (incluindo espaços, tabs e quebras de linha).
# -------------------------------------------------------------------
def limpar_strings_regex(df, colunas):
    """ Usa expressões regulares para normalizar colunas de texto:
    - Colapsa múltiplos espaços internos em um único espaço (re.sub)
    - Remove espaços nas pontas da string (.strip())
    - Preserva células nulas sem lançar erro (pd.notna) """
    df = df.copy()  # Não modifica o DataFrame original
    for col in colunas:
        df[col] = df[col].apply(
            # pd.notna(s): verifica se o valor NÃO é nulo antes de processar
            # re.sub(r"\s+", " ", str(s)): substitui sequências de espaços por um único espaço
            # .strip(): remove espaços residuais nas pontas
            lambda s: re.sub(r"\s+", " ", str(s)).strip() if pd.notna(s) else s
        )
    return df

def limpar_dados(df):
    """ Limpa o DataFrame de vendas em quatro etapas:
    1. Normaliza strings com regex (espaços extras)
    2. Converte datas e remove registros com datas inválidas
    3. Remove linhas com valores nulos em colunas obrigatórias
    4. Garante os tipos numéricos corretos
    Retorna: (df_limpo, relatorio) - o relatório documenta o impacto de cada etapa.
    """
    df = df.copy()
    n_inicial = len(df)
    relatorio = {}

    # --- Etapa 1: limpeza de strings com regex ---
    # select_dtypes("object") seleciona apenas colunas de texto
    colunas_texto = df.select_dtypes(include="object").columns
    df = limpar_strings_regex(df, colunas_texto)

    # --- Etapa 2: conversão de datas ---
    # errors="coerce" transforma valores inválidos (ex: "31/02/2024") em NaT em vez de lançar um erro - depois removemos essas linhas com dropna
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    relatorio["datas_invalidas_removidas"] = int(df["data_venda"].isnull().sum())
    df = df.dropna(subset=["data_venda"])

    # --- Etapa 3: remoção de nulos em colunas obrigatórias ---
    # Uma linha sem quantidade ou preço não pode contribuir para nenhuma
    # métrica; por isso optamos por remover (em vez de imputar) esses
    # registros.
    n_antes = len(df)
    df = df.dropna(subset=["quantidade", "preco_unitario"])
    relatorio["linhas_nulas_removidas"] = n_antes - len(df)

    # --- Etapa 4: garantia de tipos numéricos ---  # Após  o dropna, os
    # valores existentes podem ainda estar como float/object; forçamos int
    # para quantidade e float para preço.
    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    # --- Relatório final ---
    relatorio["registros_iniciais"] = n_inicial
    relatorio["registros_finais"] = len(df)
    relatorio["registros_removidos_total"] = n_inicial - len(df)  # soma de
    # todas as remoções
    print("=== RELATORIO DE LIMPEZA ===")
    for k, v in relatorio.items():
        print(f"  {k} : {v}  ")
    return df, relatorio

# EXECUÇÃO: limpar o dataset bruto e salvar como versão v1 (com
# outliers)
# Nesta versão os outliers são MANTIDOS - eles serão tratados na RF04.
df_v1, relatorio = limpar_dados(df_bruto)
os.makedirs("data/processed/v1_com_outliers", exist_ok=True)  # cria o
# diretório se não existir
df_v1.to_csv("data/processed/v1_com_outliers/vendas_v1.csv",
        index=False)
print("\nv1 salva em data/processed/v1_com_outliers/")
df_v1.head()

=== RELATORIO DE LIMPEZA ===
  datas_invalidas_removidas : 4  
  linhas_nulas_removidas : 6  
  registros_iniciais : 150  
  registros_finais : 140  
  registros_removidos_total : 10  

v1 salva em data/processed/v1_com_outliers/


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2024-01-13,Cliente_024,Mouse,Periféricos,Norte,2,120.0
3,4,2024-06-23,Cliente_013,Mouse,Periféricos,Sudeste,7,120.0
4,5,2024-11-05,Cliente_030,Tablet,Celulares,Centro-Oeste,6,1800.0
5,6,2024-05-30,Cliente_023,Notebook,Computadores,Sudeste,9,3500.0
6,7,2024-05-28,Cliente_015,Notebook,Computadores,Nordeste,4,3500.0


RF04 – Detectar e Tratar Outliers (versões v1 e v2)